<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/Nam-Wan/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_book_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import csv
import io
import random
from datetime import datetime

# รายชื่อ และ นามสกุล สำหรับลูกค้า
FIRST_NAMES = ["กิตติพงษ์", "ณิชา", "ธนกฤต", "ปรียา", "พงศกร", "ภัทรวดี", "วรวุฒิ", "ศิริพร", "อัครพล", "อนันดา"]
LAST_NAMES = ["ใจดี", "เจริญสุข", "สมบูรณ์", "วงษ์สุวรรณ", "รัตนไพศาล", "พงษ์พาณิชย์", "ชินวัตร", "ทองแท้", "สุวรรณรัตน์", "มั่นคง"]

# คลังรายชื่อหนังสือจริงสำหรับระบบ
BOOK_DATABASE = [
    {"isbn": "B001", "title": "ผ่าพิภพไททัน / Attack on Titan", "price": 90},
    {"isbn": "B002", "title": "ดาบพิฆาตอสูร / Demon Slayer", "price": 95},
    {"isbn": "B003", "title": "วันพีซ / One Piece", "price": 85},
    {"isbn": "B004", "title": "มหาศึกผนึกเทวา / Jujutsu Kaisen", "price": 95},
    {"isbn": "B005", "title": "เซเลอร์มูน / Sailor Moon", "price": 120},
    {"isbn": "B006", "title": "คิดจะพักให้คิดถึงจิตวิทยา", "price": 250},
    {"isbn": "B007", "title": "Atomic Habits เพราะหยดน้ำรวมกันเป็นมหาสมุทร", "price": 280},
    {"isbn": "B008", "title": "จิตวิทยาว่าด้วยเงิน / The Psychology of Money", "price": 290},
    {"isbn": "B009", "title": "เจ้าชายน้อย / The Little Prince", "price": 175},
    {"isbn": "B010", "title": "เซเปียนส์ ประวัติศาสตร์ย่อมนุษยชาติ", "price": 530},
    {"isbn": "B011", "title": "ปาฏิหาริย์ร้านขายยาของนามิยะ", "price": 295},
    {"isbn": "B012", "title": "สืบคดีคืนซากุระบาน", "price": 240}
]

# ==========================================
# 1. Class LibraryOrder (จัดการสมาชิก)
# ==========================================
class LibraryOrder:
    def __init__(self, name, customer_id, phone, email, points, register_date):
        self.name = name
        self.customer_id = customer_id
        self.phone = phone
        self.email = email
        self.points = int(points)
        self.register_date = register_date
        self.status = 'รอดำเนินการ'

    def register(self):
        print(f"สมัครสมาชิกสำเร็จ: {self.register_date}")

    def login(self):
        self.status = 'เข้าสู่ระบบเรียบร้อย'
        print(f"{self.name} {self.status}")
        return True

    def update_profile(self, name: str = None, phone: str = None, email: str = None):
        if name: self.name = name
        if phone: self.phone = phone
        if email: self.email = email
        print("อัปเดตข้อมูลส่วนตัวเรียบร้อย")

    def add_points(self, amount: int):
        self.points += amount
        print(f"เพิ่ม {amount} แต้ม | แต้มรวมปัจจุบัน: {self.points}")

    def get_purchase_history(self):
        return []

# ==========================================
# 2. Class Book (จัดการข้อมูลหนังสือ)
# ==========================================
class Book:
    def __init__(self, book_isbn, book_title, book_author, price, category_id, sub_category_id, shelf_location, stock_qty):
        self.book_isbn = book_isbn
        self.book_title = book_title
        self.book_author = book_author
        self.price = float(price)
        self.category_id = category_id
        self.sub_category_id = sub_category_id
        self.shelf_location = shelf_location
        self.stock_qty = int(stock_qty)

# ==========================================
# 3. Class Receipt (ระบบออกใบเสร็จและคิดเงิน)
# ==========================================
class Receipt:
    def __init__(self, order_id: str, customer: LibraryOrder, items: list, days_rented: int, days_late: int = 0):
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.days_rented = days_rented
        self.days_late = max(0, days_late)
        self.date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def _calculate_book_rental_fee(self, days: int) -> float:
        """คำนวณค่ายืมต่อเล่ม: 3 วัน 20 บาท, เศษวันละ 7 บาท"""
        sets_of_3 = days // 3
        remaining_days = days % 3
        return (sets_of_3 * 20) + (remaining_days * 7)

    def calculate_totals(self):
        total_books = len(self.items)

        # แต้มสะสม: ยืม 1 เล่ม = 1 แต้ม, คืนตรงเวลา +1 แต้ม
        base_points = total_books
        on_time_bonus = 1 if self.days_late == 0 else 0
        total_points_accumulated = self.customer.points + base_points + on_time_bonus

        # ส่วนลด: ครบ 10 แต้ม ได้สิทธิ์อ่านฟรี 3 วัน (20 บาท/เล่ม)
        free_books_count = total_points_accumulated // 10
        free_books_count = min(free_books_count, total_books)

        # คำนวณค่ายืมปกติ
        rental_fee_per_book = self._calculate_book_rental_fee(self.days_rented)
        total_rental_before_discount = total_books * rental_fee_per_book

        # คำนวณส่วนลดสิทธิ์ยืมฟรี 3 วัน (หักสูงสุด 20 บาท ต่อเล่ม)
        discount_per_book = min(20.0, rental_fee_per_book)
        total_discount = free_books_count * discount_per_book

        # สรุปค่ายืมหลังหักส่วนลด
        total_rental_fee = total_rental_before_discount - total_discount

        # คำนวณค่าปรับ (คืนช้า: 10 บาท / เล่ม / วัน)
        total_fine = total_books * self.days_late * 10

        # สรุปรวมเงิน
        grand_total = total_rental_fee + total_fine
        earned_points = base_points + on_time_bonus

        return {
            "total_books": total_books,
            "base_points": base_points,
            "on_time_bonus": on_time_bonus,
            "total_points_accumulated": total_points_accumulated,
            "free_books_count": free_books_count,
            "total_discount": total_discount,
            "rental_fee_per_book": rental_fee_per_book,
            "total_rental_fee": total_rental_fee,
            "total_fine": total_fine,
            "grand_total": grand_total,
            "earned_points": earned_points
        }

    def print_receipt(self):
        calc = self.calculate_totals()

        # อัปเดตแต้มคงเหลือของลูกค้า
        points_used = calc["free_books_count"] * 10
        net_new_points = calc["earned_points"] - points_used
        self.customer.points += net_new_points

        print("=" * 55)
        print(f"{'ใบเสร็จรับเงิน / Receipt':^55}")
        print("=" * 55)
        print(f"เลขที่ใบเสร็จ: {self.order_id}")
        print(f"วันที่ออกใบเสร็จ: {self.date_issued}")
        print(f"ชื่อลูกค้า: {self.customer.name} (ID: {self.customer.customer_id})")
        print(f"จำนวนวันที่ยืม: {self.days_rented} วัน")
        print(f"สถานะการคืน: {'⚠️ คืนช้า ' + str(self.days_late) + ' วัน' if self.days_late > 0 else '✅ คืนตรงเวลา'}")
        print("-" * 55)

        print(f"รายการหนังสือที่ยืม ({calc['total_books']} เล่ม):")
        for i, book in enumerate(self.items, 1):
            print(f"  [{i:02d}] {book.book_title} (ราคาปก {book.price:.0f} บาท)")

        print("-" * 55)
        print(f"อัตราค่ายืมปกติต่อเล่ม ({self.days_rented} วัน): {calc['rental_fee_per_book']:.2f} บาท")

        if calc['free_books_count'] > 0:
            print(f"🎁 ส่วนลดสะสมแต้ม (ครบ 10 แต้ม): ฟรี 3 วัน จำนวน {calc['free_books_count']} เล่ม (-{calc['total_discount']:.2f} บาท)")

        print(f"รวมค่ายืมหนังสือหลังหักส่วนลด: {calc['total_rental_fee']:.2f} บาท")

        if calc['total_fine'] > 0:
            print(f"❌ ค่าปรับคืนช้า ({self.days_late} วัน x {calc['total_books']} เล่ม x 10B): {calc['total_fine']:.2f} บาท")

        print("-" * 55)
        print(f"ยอดชำระสุทธิ (Grand Total): {calc['grand_total']:.2f} บาท")
        print("-" * 55)

        print(f"✨ แต้มที่ได้รับจากจำนวนหนังสือ (1 เล่ม = 1 แต้ม): +{calc['base_points']} แต้ม")
        if self.days_late == 0:
            print(f"🎉 โบนัสคืนตรงเวลา: +{calc['on_time_bonus']} แต้ม")
        else:
            print("🚫 คืนช้ากว่ากำหนด: ไม่ได้รับโบนัสคืนตรงเวลา (+0 แต้ม)")

        if points_used > 0:
            print(f"🔄 ใช้แต้มแลกอ่านฟรี: -{points_used} แต้ม")

        print(f"🏆 แต้มสะสมคงเหลือปัจจุบัน: {self.customer.points} แต้ม")
        print("=" * 55 + "\n")

# ==========================================
# 4. ประมวลผลและออกใบเสร็จ
# ==========================================

# 1. ลูกค้า
random_name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
user1 = LibraryOrder(
    name=random_name,
    customer_id=f"C{random.randint(100, 999)}",
    phone="081-234-5678",
    email="user@email.com",
    points=random.choice([0, 5, 8, 10, 12]),
    register_date="2026-01-01"
)

# 2. เลือกหนังสือหลากหลายเล่มจากคลัง
num_books = random.randint(1, 6)
selected_books_data = random.sample(BOOK_DATABASE, k=num_books)

cart_items = []
for b_data in selected_books_data:
    b = Book(
        book_isbn=b_data["isbn"],
        book_title=b_data["title"],
        book_author="ไม่ระบุผู้แต่ง",
        price=b_data["price"],
        category_id="01",
        sub_category_id="01",
        shelf_location="A-01",
        stock_qty=10
    )
    cart_items.append(b)

# 3. กำหนดระยะเวลายืม
days_rented = random.choice([3, 5, 7, 10])
days_late = random.choices([0, 1, 2, 3], weights=[0.6, 0.2, 0.1, 0.1])[0]

# 4. พิมพ์ใบเสร็จ
receipt1 = Receipt(
    order_id=f"REC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000, 9999)}",
    customer=user1,
    items=cart_items,
    days_rented=days_rented,
    days_late=days_late
)
receipt1.print_receipt()

               ใบเสร็จรับเงิน / Receipt                
เลขที่ใบเสร็จ: REC-20260829-3787
วันที่ออกใบเสร็จ: 2026-08-29 10:56:48
ชื่อลูกค้า: ภัทรวดี ทองแท้ (ID: C883)
จำนวนวันที่ยืม: 10 วัน
สถานะการคืน: ✅ คืนตรงเวลา
-------------------------------------------------------
รายการหนังสือที่ยืม (1 เล่ม):
  [01] เซเปียนส์ ประวัติศาสตร์ย่อมนุษยชาติ (ราคาปก 530 บาท)
-------------------------------------------------------
อัตราค่ายืมปกติต่อเล่ม (10 วัน): 67.00 บาท
รวมค่ายืมหนังสือหลังหักส่วนลด: 67.00 บาท
-------------------------------------------------------
ยอดชำระสุทธิ (Grand Total): 67.00 บาท
-------------------------------------------------------
✨ แต้มที่ได้รับจากจำนวนหนังสือ (1 เล่ม = 1 แต้ม): +1 แต้ม
🎉 โบนัสคืนตรงเวลา: +1 แต้ม
🏆 แต้มสะสมคงเหลือปัจจุบัน: 2 แต้ม

